# OLMo 2 7B HumanEval: Completion-Bag Mixtures

HumanEval-only pass@k curves for the reviewer interpretation where each completion is sampled from either the canonical temperature bag or the pooled retokenization bag. For a task `i` and mixture weight `gamma`, the effective failure probability is

$$p^{mix}_{fail,i}(\gamma) = (1 - \gamma)p^{canon}_{fail,i} + \gamma p^{retok}_{fail,i}.$$

The corresponding curve is

$$pass@k(\gamma) = \frac{1}{N}\sum_i \left(1 - \left(p^{mix}_{fail,i}(\gamma)\right)^k\right).$$

This notebook uses all pooled retokenization completions across `p` values for `pass@retok`; individual retokenization probabilities are intentionally ignored.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "figure_notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from eval import load

FIGURE_DIR = REPO_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "svg.fonttype": "none",
})


In [ ]:
MODEL_NAME = "allenai/OLMo-2-1124-7B-Instruct"
MODEL_LABEL = "OLMo 2 7B Instruct"
DATASET = "humaneval"
DATASET_SIZE = 164
NUM_VARIANTS = 51
MAX_K = 50
GAMMAS = [0.25, 0.50, 0.75]


In [ ]:
def normalize_outcomes(df: pd.DataFrame, *, dataset: str = DATASET) -> pd.DataFrame:
    df = df.copy()
    if "passed" not in df.columns and "Correct" in df.columns:
        df["passed"] = df["Correct"]
    if "passed" not in df.columns and {"generated_answer", "answer"}.issubset(df.columns):
        df["passed"] = (df["generated_answer"] == df["answer"]).astype(int)
    if "passed" not in df.columns:
        raise ValueError("Expected a passed/Correct column or generated_answer+answer columns.")
    if "task_id" not in df.columns and "prompti" in df.columns:
        df["task_id"] = df["prompti"].map(lambda i: f"{dataset}/{i}")
    if "task_id" not in df.columns:
        raise ValueError("Expected task_id or prompti column.")
    df["passed"] = df["passed"].astype(int)
    return df


def task_failure_probabilities(df: pd.DataFrame, *, label: str) -> pd.DataFrame:
    summary = (
        normalize_outcomes(df)
        .groupby("task_id", sort=True)["passed"]
        .agg(num_samples="size", num_correct="sum", pass_prob="mean")
        .reset_index()
    )
    summary[f"p_fail_{label}"] = 1.0 - summary["pass_prob"]
    return summary[["task_id", "num_samples", "num_correct", f"p_fail_{label}"]].rename(
        columns={
            "num_samples": f"num_samples_{label}",
            "num_correct": f"num_correct_{label}",
        }
    )


def pass_curve_from_failure_probabilities(p_fail: np.ndarray, *, max_k: int = MAX_K) -> pd.DataFrame:
    ks = np.arange(1, max_k + 1, dtype=int)
    pass_rates_by_task = 1.0 - np.power(p_fail[:, None], ks[None, :])
    return pd.DataFrame({
        "k": ks,
        "pass_rate": pass_rates_by_task.mean(axis=0),
        "pass_rate_std": pass_rates_by_task.std(axis=0, ddof=1) / np.sqrt(pass_rates_by_task.shape[0]),
    })


def curve_row(curves: dict[str, pd.DataFrame], label: str, k: int) -> float:
    curve = curves[label]
    return float(curve.loc[curve["k"] == k, "pass_rate"].iloc[0])


In [ ]:
df_canon = load.load_humaneval(
    model_name=MODEL_NAME,
    dataset_size=DATASET_SIZE,
    numvariants=NUM_VARIANTS,
    variant_type="temperature",
)
df_retok = load.load_humaneval(
    model_name=MODEL_NAME,
    dataset_size=DATASET_SIZE,
    numvariants=NUM_VARIANTS,
    variant_type="retok",
)

canon_fail = task_failure_probabilities(df_canon, label="canon")
retok_fail = task_failure_probabilities(df_retok, label="retok")
failure = canon_fail.merge(retok_fail, on="task_id", how="inner")

expected_tasks = DATASET_SIZE
if len(failure) != expected_tasks:
    raise ValueError(f"Expected {expected_tasks} aligned tasks, found {len(failure)}.")

failure.head()


In [ ]:
fig, ax = plt.subplots()

ax.scatter(failure['p_fail_canon'], failure['p_fail_retok'])
ax.set_xlim([-0.1,1.1])
ax.set_ylim([-0.1,1.1])
ax.set_xlabel('p_fail canon')
ax.set_xlabel('p_fail retok')

In [ ]:
    curves = {
    "pass@k": pass_curve_from_failure_probabilities(failure["p_fail_canon"].to_numpy(dtype=float)),
    "pass@retok": pass_curve_from_failure_probabilities(failure["p_fail_retok"].to_numpy(dtype=float)),
}

for gamma in GAMMAS:
    p_fail_mix = (
        (1.0 - gamma) * failure["p_fail_canon"].to_numpy(dtype=float)
        + gamma * failure["p_fail_retok"].to_numpy(dtype=float)
    )
    curves[f"gamma={gamma:.2f}"] = pass_curve_from_failure_probabilities(p_fail_mix)

summary_rows = []
for label, curve in curves.items():
    if label == "pass@k":
        mean_p_fail = failure["p_fail_canon"].mean()
    elif label == "pass@retok":
        mean_p_fail = failure["p_fail_retok"].mean()
    else:
        gamma = float(label.split("=")[1])
        mean_p_fail = ((1.0 - gamma) * failure["p_fail_canon"] + gamma * failure["p_fail_retok"]).mean()
    summary_rows.append({
        "curve": label,
        "mean_p_fail": mean_p_fail,
        "pass@1": curve_row(curves, label, 1),
        "pass@10": curve_row(curves, label, 10),
        "pass@50": curve_row(curves, label, 50),
    })

summary = pd.DataFrame(summary_rows)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)

styles = {
    "pass@k": {"color": "black", "linestyle": "-", "linewidth": 2.5},
    "pass@retok": {"color": "firebrick", "linestyle": "--", "linewidth": 2.5},
    "gamma=0.25": {"color": "#2f6f95", "linestyle": "-.", "linewidth": 2.0},
    "gamma=0.50": {"color": "#5f8d3b", "linestyle": "-.", "linewidth": 2.0},
    "gamma=0.75": {"color": "#b8872f", "linestyle": "-.", "linewidth": 2.0},
}

labels = {
    "pass@k": "pass@k",
    "pass@retok": "pass@retok",
    "gamma=0.25": r"mix $\gamma=0.25$",
    "gamma=0.50": r"mix $\gamma=0.50$",
    "gamma=0.75": r"mix $\gamma=0.75$",
}

for label in ["pass@k", "gamma=0.25", "gamma=0.50", "gamma=0.75", "pass@retok"]:
    curve = curves[label]
    style = styles[label]
    ax.fill_between(
        curve["k"],
        curve["pass_rate"] - curve["pass_rate_std"],
        curve["pass_rate"] + curve["pass_rate_std"],
        color=style["color"],
        alpha=0.06,
        linewidth=0,
    )
    ax.plot(
        curve["k"],
        curve["pass_rate"],
        label=labels[label],
        **style,
    )

ax.set_title(f"HumanEval completion-bag mixtures: {MODEL_LABEL}")
ax.set_xlim(1, MAX_K)
ax.set_ylim(0, 1)
ax.set_xlabel("k")
ax.set_ylabel("Pass Rate")
ax.grid(True, alpha=0.3)
ax.legend(frameon=True, loc="lower right")

out_svg = FIGURE_DIR / "humaneval_olmo2_temp_retok_completion_mixture_passk.svg"
out_png = FIGURE_DIR / "humaneval_olmo2_temp_retok_completion_mixture_passk.png"
fig.savefig(out_svg, bbox_inches="tight")
fig.savefig(out_png, bbox_inches="tight")
out_svg


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5), constrained_layout=True)

bins = np.linspace(0, 1, 25)
p_fail_by_label = {
    "pass@k": failure["p_fail_canon"].to_numpy(dtype=float),
    "pass@retok": failure["p_fail_retok"].to_numpy(dtype=float),
}
for gamma in GAMMAS:
    p_fail_by_label[f"gamma={gamma:.2f}"] = (
        (1.0 - gamma) * failure["p_fail_canon"].to_numpy(dtype=float)
        + gamma * failure["p_fail_retok"].to_numpy(dtype=float)
    )

for label in ["pass@k", "gamma=0.25", "gamma=0.50", "gamma=0.75", "pass@retok"]:
    p_fail = p_fail_by_label[label]
    counts, hist_bins = np.histogram(p_fail, bins=bins)
    counts = counts / counts.sum()
    ax.step(
        hist_bins,
        np.append(counts, counts[-1]),
        where="post",
        label=labels[label],
        color=styles[label]["color"],
        linestyle=styles[label]["linestyle"],
        linewidth=styles[label]["linewidth"],
    )
    ax.axvline(
        p_fail.mean(),
        color=styles[label]["color"],
        linestyle=styles[label]["linestyle"],
        linewidth=1,
        alpha=0.6,
    )

ax.set_title(f"HumanEval failure probabilities: {MODEL_LABEL}")
ax.set_xlabel(r"$P_{fail}$")
ax.set_ylabel("Fraction of Tasks")
ax.set_xticks(np.linspace(0, 1, 6))
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

out_svg = FIGURE_DIR / "humaneval_olmo2_temp_retok_completion_mixture_pfail.svg"
out_png = FIGURE_DIR / "humaneval_olmo2_temp_retok_completion_mixture_pfail.png"
fig.savefig(out_svg, bbox_inches="tight")
fig.savefig(out_png, bbox_inches="tight")
out_svg


## Gamma Slice

Sweep `gamma` at a fixed `k` to check whether the completion-bag mixture is monotonic, especially near small nonzero mixture weights. The x-axis uses a symmetric log scale so `gamma=0` can be shown alongside log-spaced positive values.

In [ ]:
MIXTURE_SLICE_K = 50
GAMMA_LINTHRESH = 1e-3
GAMMA_SWEEP = np.unique(np.concatenate([
    np.array([0.0]),
    np.logspace(np.log10(GAMMA_LINTHRESH), 0, 200),
]))


def pass_at_k_for_gamma(gamma: float, *, k: int = MIXTURE_SLICE_K) -> float:
    p_fail_mix = (
        (1.0 - gamma) * failure["p_fail_canon"].to_numpy(dtype=float)
        + gamma * failure["p_fail_retok"].to_numpy(dtype=float)
    )
    return float(np.mean(1.0 - np.power(p_fail_mix, k)))


gamma_slice = pd.DataFrame({
    "gamma": GAMMA_SWEEP,
    f"pass@{MIXTURE_SLICE_K}": [pass_at_k_for_gamma(gamma) for gamma in GAMMA_SWEEP],
})

display_points = np.unique(np.concatenate([
    np.array([0.0]),
    np.logspace(np.log10(GAMMA_LINTHRESH), 0, 25),
]))
display_table = pd.DataFrame({
    "gamma": display_points,
    f"pass@{MIXTURE_SLICE_K}": [pass_at_k_for_gamma(gamma) for gamma in display_points],
})
display_table


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)


y_col = f"pass@{MIXTURE_SLICE_K}"
ax.plot(
    gamma_slice["gamma"],
    gamma_slice[y_col],
    color="black",
    linewidth=2.25,
)
ax.scatter(
    [0.0, 0.25, 0.50, 0.75, 1.0],
    [pass_at_k_for_gamma(gamma) for gamma in [0.0, 0.25, 0.50, 0.75, 1.0]],
    color=["black", "#2f6f95", "#5f8d3b", "#b8872f", "firebrick"],
    s=42,
    zorder=3,
)

ax.set_xscale("symlog", linthresh=GAMMA_LINTHRESH, linscale=0.6)
ax.set_yscale('log')
ax.set_xlim(0, 1)
ax.set_xticks([0.0, GAMMA_LINTHRESH, 0.01, 0.1, 1.0])
ax.set_xticklabels(["0", f"{GAMMA_LINTHRESH:g}", "0.01", "0.1", "1"])
ax.set_xlabel(r"Mixture weight $\gamma$")
ax.set_ylabel(y_col)
ax.set_title(f"HumanEval completion-bag mixture slice at k={MIXTURE_SLICE_K}")
ax.grid(True, alpha=0.3, which="both")

out_svg = FIGURE_DIR / f"humaneval_olmo2_temp_retok_completion_mixture_gamma_slice_k{MIXTURE_SLICE_K}.svg"
out_png = FIGURE_DIR / f"humaneval_olmo2_temp_retok_completion_mixture_gamma_slice_k{MIXTURE_SLICE_K}.png"
fig.savefig(out_svg, bbox_inches="tight")
fig.savefig(out_png, bbox_inches="tight")
out_svg
